# 01 — Screen Capture & Perception Tests

Validates the full Perception Loop pipeline:
1. Screenshot capture via `mss`
2. Perceptual-hash deduplication
3. VLM call (`gemma4:latest`) → raw JSON
4. `GameStateSnapshot` parsing
5. Deterministic alarm checks

> **Read-only test** — no game input is sent.

In [ ]:
import sys
sys.path.insert(0, "/mnt/e/Personal/Samarth/repository/AOE-Agent/src")

from PIL import Image
from pathlib import Path

TEST_IMAGE = Path("/mnt/e/Personal/Samarth/repository/AOE-Agent/old/tests/test_image.jpg")
img = Image.open(TEST_IMAGE)
print(f"Test image loaded: {img.size} {img.mode}")

## 1 — Screenshot Capture

In [ ]:
from aoe2_agent.perception.capture import capture_screenshot, get_aoe2_window_bbox

bbox = get_aoe2_window_bbox()
print(f"AoE2 window bbox: {bbox}  (None = full-screen fallback)")

live = capture_screenshot(bbox=bbox)
print(f"Captured: {live.size} {live.mode}")
live

## 2 — Perceptual-Hash Deduplication

In [ ]:
import imagehash

def phash(img):
    return imagehash.average_hash(img.resize((64, 64)).convert("L"))

# Identical images → diff == 0
h1 = phash(img)
h2 = phash(img)
print(f"Same image diff:     {h1 - h2}  (expect 0)")

# Live screen vs test image → should differ
h3 = phash(live)
print(f"Test vs live diff:   {h1 - h3}  (expect > 0 unless AoE2 is open)")

THRESHOLD = 8
needs_vlm = (h1 - h3) > THRESHOLD
print(f"needs_vlm (diff > {THRESHOLD}): {needs_vlm}")

In [ ]:
# CaptureLoop end-to-end: two consecutive captures should deduplicate
from aoe2_agent.perception.capture import CaptureLoop

loop = CaptureLoop(out_dir="/tmp/aoe2_test_caps", interval=1.0, hash_threshold=8)

path1, img1, needs1 = loop.run_once()
path2, img2, needs2 = loop.run_once()  # second capture seconds later — likely identical

print(f"Capture 1: {path1}  needs_vlm={needs1}")
print(f"Capture 2: {path2}  needs_vlm={needs2}  (likely False — same frame)")

## 3 — VLM Call: gemma4:latest on Test Image

In [ ]:
import io, base64
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage
from aoe2_agent.perception.loop import VISION_SYSTEM_PROMPT, _img_to_b64

llm = ChatOllama(model="gemma4:latest", temperature=0)

b64 = _img_to_b64(img)

messages = [
    SystemMessage(content=VISION_SYSTEM_PROMPT),
    HumanMessage(
        content=[
            {"type": "text", "text": "Analyze this Age of Empires II screenshot."},
            {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{b64}"}},
        ]
    ),
]

response = llm.invoke(messages)
raw = response.content
print(raw)

## 4 — Parse VLM Output → GameStateSnapshot

In [ ]:
from aoe2_agent.perception.loop import _parse_snapshot

snap = _parse_snapshot(raw)
print(snap)

In [ ]:
# Inspect individual fields
if snap:
    print(f"Age:             {snap.age}")
    print(f"Population:      {snap.population}/{snap.pop_cap}")
    print(f"Resources:       {snap.resources}")
    print(f"Threat level:    {snap.threat_level}")
    print(f"Idle villagers:  {snap.idle_villagers_visible}")
    print(f"Buildings:       {snap.visible_buildings}")
    print(f"Notes:           {snap.notes}")

    print("\nVisible units:")
    for u in snap.visible_units:
        print(f"  {u.approx_count}x {u.label} ({u.owner})")
else:
    print("Parse failed — check raw output above")

## 5 — Deterministic Alarm Checks

In [ ]:
from collections import deque
from aoe2_agent.alarms.rules import check_alarms
from aoe2_agent.schemas import GameStateSnapshot, DetectedUnit

# Synthetic low-food, idle-villager, high-threat snapshot
danger_snap = GameStateSnapshot(
    age="Feudal",
    population=28,
    pop_cap=30,
    resources={"food": 120, "wood": 80, "gold": 10, "stone": 200},
    visible_units=[
        DetectedUnit(label="knight", owner="enemy", approx_count=5),
    ],
    visible_buildings=["Town Center"],
    idle_villagers_visible=True,
    threat_level="high",
    notes="Knights approaching the Town Center",
)

alarms = check_alarms(danger_snap)
print(f"{len(alarms)} alarms fired:")
for a in alarms:
    print(f"  ⚠  {a}")

In [ ]:
# Healthy snapshot — no alarms expected
healthy_snap = GameStateSnapshot(
    age="Castle",
    population=80,
    pop_cap=100,
    resources={"food": 600, "wood": 400, "gold": 300, "stone": 150},
    visible_units=[DetectedUnit(label="villager", owner="self", approx_count=20)],
    visible_buildings=["Town Center", "Castle", "Barracks"],
    idle_villagers_visible=False,
    threat_level="none",
    notes="",
)

alarms = check_alarms(healthy_snap)
print(f"Alarms on healthy snapshot: {alarms}  (expect [])")

In [ ]:
# Trend alarm: rapid food drop across history
h = deque(maxlen=20)
for food in [900, 750, 600]:  # history snapshots
    h.append(GameStateSnapshot(
        age="Castle", population=80, pop_cap=100,
        resources={"food": food, "wood": 300, "gold": 200, "stone": 100},
        visible_units=[], visible_buildings=[],
        idle_villagers_visible=False, threat_level="none", notes="",
    ))

current = GameStateSnapshot(
    age="Castle", population=80, pop_cap=100,
    resources={"food": 280, "wood": 300, "gold": 200, "stone": 100},
    visible_units=[], visible_buildings=[],
    idle_villagers_visible=False, threat_level="none", notes="",
)

alarms = check_alarms(current, h)
print(f"Trend alarms: {alarms}")

## 6 — Full Perception Loop (single async tick)

In [ ]:
import asyncio
from aoe2_agent.perception.loop import PerceptionLoop

received = []
alarm_log = []

loop_obj = PerceptionLoop(
    on_snapshot=lambda s: received.append(s),
    on_alarm=lambda a: alarm_log.extend(a),
)

# Run a single tick by calling the VLM directly on our test image
snap = await loop_obj._call_vlm(img)

if snap:
    received.append(snap)
    print("Snapshot received:")
    print(snap.model_dump_json(indent=2))
else:
    print("No snapshot — VLM parsing failed")